In [14]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# 1. Charge les variables cachées dans le fichier .env
load_dotenv()

# 2. Récupère le jeton
mon_token = os.getenv("HF_TOKEN")

# 3. Authentifie la session courante
login(token=mon_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [20]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct"
 )
#template_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [117]:
template_tokenizer.pad_token

In [21]:
template_tokenizer

TokenizersBackend(name_or_path='meta-llama/Llama-3.2-1B-Instruct', vocab_size=128000, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False, normalize

In [101]:
from pathlib import Path

patterns = [
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt"
]

# 1. On charge d'abord tous vos fichiers dans un seul bloc global
dataset_complet = load_dataset("text", data_files=patterns, split="train", sample_by= "document")

# récupérer les fichiers dans le même ordre
files = []
for p in patterns:
    files.extend(sorted(Path().glob(p)))

def add_id(example, idx):
    path = files[idx]
    example["id"] = path.stem
    example["annee"] = path.parts[-3]
    example["election"] = path.parts[-2]
    return example

dataset_complet = dataset_complet.map(add_id, with_indices=True)



Resolving data files:   0%|          | 0/12498 [00:00<?, ?it/s]

In [102]:
def add_soutien(example) : 
    # print(metadata[metadata['id'] == example['id']]['titulaire-soutien'].values)
    # print(example['id'])
    example['soutien'] = metadata[metadata['id'] == example['id']]['titulaire-soutien'].values[0]
    return example

dataset_complet = dataset_complet.map(add_soutien)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [98]:
dataset_complet

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien'],
    num_rows: 12498
})

In [99]:
dataset_complet[0]

{'text': "ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\nLes élections législatives sont une étape importante pour la consolida- tion de cette victoire : il faut une majorité de gauche à l'Assemblée Nationale.\nmaire-adjointe à Bourg-en-bresse assistante sociale\n... IL FAUT QUE TOUTES LES FORCES DE GAUCHE ...\nUne condition essentielle de la victoire est la reconnaissance du plura- lisme respectant les différences des forces politiques de gauche.\nCela est nécessaire pour éviter des alliances centristes qui réduiraient à néant la victoire du 10 mai.\nDans cet esprit, le P.S.U. entend au premier tour de ces élections met- tre l'accent sur des questions essentielles qui n'ont reçu, de la part de la gauche traditionnelle (P.S. et P.C.F.), que des réponses évasives ou des re

In [ ]:
# 2. On divise ce bloc (ici : 10 % pour le test, 90 % pour l'entraînement)
datasets_divises = dataset_complet.train_test_split(test_size=0.1, seed=42)

# 3. On extrait nos deux sous-ensembles prêts à l'emploi !
dataset_train = datasets_divises["train"]
dataset_test = datasets_divises["test"]

In [34]:
import pandas as pd
metadata = pd.read_csv("data/archelect_search.csv")

In [ ]:
metadata[metadata['id'] == "EL134_L_1981_06_001_01_1_PF_01"]['titulaire-soutien'].item()

'Parti socialiste unifié'

In [93]:
metadata[metadata['id'] == "EL177_L_1988_06_094_04_1_PF_01"]['titulaire-nom']

5684    Jégou
Name: titulaire-nom, dtype: object

In [92]:
metadata[metadata['id'] == "EL177_L_1988_06_094_04_1_PF_02"]['titulaire-nom']

5685    Schénardi
Name: titulaire-nom, dtype: object

In [94]:
metadata[metadata['id'] == "EL177_L_1988_06_094_04_1_PF_03"]['titulaire-nom']

5686    Hédouin
Name: titulaire-nom, dtype: object

In [95]:
metadata[metadata['id'] == "EL177_L_1988_06_094_04_1_PF_04"]['titulaire-nom']

5687    Delaporte
Name: titulaire-nom, dtype: object

In [90]:
metadata[metadata['titulaire-nom'] == 'Delaporte']

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
5687,EL177_L_1988_06_094_04_1_PF_04,1988-06-05,Assemblée Nationale;Ve République;Élections lé...,"Élections législatives de 1988, Val-de-Marne -...",législatives,1,EL177,94,Val-de-Marne,94 - Val-de-Marne,...,48,entre 40 et 49 ans,non mentionné,maire,non mentionné,non mentionné,non mentionné,non mentionné,Majorité présidentielle pour la France unie,non
6621,EL177_L_1988_06_094_04_2_PF_01,1988-06-12,Ve République;France;Assemblée Nationale;Élect...,"Élections législatives de 1988, Val-de-Marne -...",législatives,2,EL177,94,Val-de-Marne,94 - Val-de-Marne,...,non mentionné,non mentionné,non mentionné,maire,non mentionné,non mentionné,non mentionné,non mentionné,Majorité présidentielle pour la France unie,non
11384,EL198_L_1993_03_094_04_1_PF_07,1993-03-21,Ve République;Élections législatives;France;As...,"Élections législatives de 1993, Val-de-Marne -...",législatives,1,EL198,94,Val-de-Marne,94 - Val-de-Marne,...,non mentionné,non mentionné,non mentionné,maire,non mentionné,non mentionné,non mentionné,non mentionné,Alliance des Français pour le progrès,non


In [36]:
metadata.columns

Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
       'cote', 'departement', 'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
      dtype='object')

In [61]:
metadata['titulaire-liste'].value_counts(normalize=True)*100

titulaire-liste
non mentionné                                       41.894703
Union du rassemblement et du centre                  6.505041
Majorité présidentielle pour la France unie          6.160986
Union pour une nouvelle majorité                     5.656905
Entente des écologistes                              4.344695
                                                      ...    
Libération                                           0.008001
Liberté égalité justice                              0.008001
Relève                                               0.008001
Union de la majorité nationale pour la France        0.008001
Rassemblement de gauche et des forces de progrès     0.008001
Name: proportion, Length: 499, dtype: float64

In [42]:
metadata['titulaire-soutien'].value_counts(normalize=True)*100

titulaire-soutien
non mentionné                                                                                                                        24.291887
Parti communiste français                                                                                                            12.257961
Front national                                                                                                                        9.865578
Parti socialiste                                                                                                                      7.913266
Rassemblement pour la République;Union pour la démocratie française                                                                   6.072972
                                                                                                                                       ...    
Union pour la démocratie française;Rassemblement pour la République;Parti républicain;Parti radical;Centre des démocrates so

In [44]:
metadata.iloc[0]

id                                                   EL134_L_1981_06_001_01_1_PF_01
date                                                                     1981-06-14
subject                           France;Assemblée Nationale;Ve République;Élect...
title                             Élections législatives de 1981, Ain - 01, circ...
contexte-election                                                      législatives
contexte-tour                                                                     1
cote                                                                          EL134
departement                                                                      01
departement-nom                                                                 Ain
departement-insee                                                          01 - Ain
identifiant de circonscription                                                    1
images                            https://ia601807.us.archive.org/31/items/E

In [103]:
def add_prompt(example) : 
    soutien = example['soutien']
    example['prompt'] = f"Rédige une profession de foi pour un candidat soutenu par le parti : {soutien}."
    return example

In [104]:
dataset_complet = dataset_complet.map(add_prompt)
dataset_complet

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prompt'],
    num_rows: 12498
})

In [105]:
dataset_complet['prompt'][0]

'Rédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.'

In [110]:
def add_messages(example) : 
    example['messages'] = [{"role" : "user", "content" : f"{example['prompt']}"},
                           {"role" : "assistant", "content" : f"{example['text']}"}]
    return example

In [111]:
dataset_complet = dataset_complet.map(add_messages)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [113]:
dataset_complet['messages']

Column([[{'content': 'Rédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.', 'role': 'user'}, {'content': "ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\nLes élections législatives sont une étape importante pour la consolida- tion de cette victoire : il faut une majorité de gauche à l'Assemblée Nationale.\nmaire-adjointe à Bourg-en-bresse assistante sociale\n... IL FAUT QUE TOUTES LES FORCES DE GAUCHE ...\nUne condition essentielle de la victoire est la reconnaissance du plura- lisme respectant les différences des forces politiques de gauche.\nCela est nécessaire pour éviter des alliances centristes qui réduiraient à néant la victoire du 10 mai.\nDans cet esprit, le P.S.U. entend au premier tour de ces élections met- tre l'accent s

In [114]:
def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""
    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

In [115]:
dataset_complet = dataset_complet.map(format_prompt)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [116]:
dataset_complet['text'][0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 14 Mar 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nRédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\nLes élections législatives sont une étape importante pour la consolida- tion de cette victoire : il faut une majorité de gauche à l'Assemblée Nationale.\nmaire-adjointe à Bourg-en-bresse assistante sociale\n... IL FAUT QUE TOUTES LES FORCES DE GAUCHE ...\nUne condition essentielle de la victoire est la reconnaissance du plura- lisme respectant les différences des forces politiques de gauche.\nCel

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# # 4-bit quantization configuration - Q in QLoRA
# bnb_config = BitsAndBytesConfig(
# load_in_4bit=True, # Use 4-bit precision model loading
# bnb_4bit_quant_type="nf4", # Quantization type
# bnb_4bit_compute_dtype="float16", # Compute dtype
# bnb_4bit_use_double_quant=True, # Apply nested quantization
# )
# # Load the model to train on the GPU
# model = AutoModelForCausalLM.from_pretrained(
# model_name,
# device_map="auto",
# # Leave this out for regular SFT
# quantization_config=bnb_config,
# )
# model.config.use_cache = False
# model.config.pretraining_tp = 1
# # Load LLaMA tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = "<PAD>"
# tokenizer.padding_side = "left"

In [118]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 1. Chargement et configuration du Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# On recycle le jeton de fin existant au lieu d'en inventer un
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "left" # Pour l'entraînement (Causal LM), on ajoute le remplissage à la fin

# 2. Chargement du Modèle (optimisé pour Apple Silicon)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="mps",           # On envoie directement sur la puce graphique de votre Mac
    torch_dtype=torch.bfloat16  # Précision 16-bit : le modèle pèsera environ 2.5 Go en RAM
)
model.config.use_cache = False
model.config.pretraining_tp = 1

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [119]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# Prepare LoRA Configuration
peft_config = LoraConfig(
lora_alpha=32, # LoRA Scaling
lora_dropout=0.1, # Dropout for LoRA Layers
r=64, # Rank
bias="none",
task_type="CAUSAL_LM",
target_modules= # Layers to target
["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
"down_proj"]
)
# Prepare model for training
#model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [121]:
from transformers import TrainingArguments
output_dir = "./results"
# Training arguments
training_arguments = TrainingArguments(
output_dir=output_dir,
per_device_train_batch_size=2,
gradient_accumulation_steps=4,
#optim="paged_adamw_32bit",
optim = "adamw_torch",
learning_rate=2e-4,
lr_scheduler_type="cosine",
num_train_epochs=1,
logging_steps=10,
#fp16=True,
bf16=True,
gradient_checkpointing=True
)

In [128]:
dataset_light = dataset_complet.remove_columns(['prompt'])

In [129]:
dataset_light

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'messages'],
    num_rows: 12498
})

In [131]:
from trl import SFTTrainer, SFTConfig
# Set supervised fine-tuning parameters
training_arguments = SFTConfig(
output_dir=output_dir,
per_device_train_batch_size=2,
gradient_accumulation_steps=4,
#optim="paged_adamw_32bit",
optim = "adamw_torch",
learning_rate=2e-4,
lr_scheduler_type="cosine",
num_train_epochs=1,
logging_steps=10,
#fp16=True,
bf16=True,
gradient_checkpointing=True,
dataset_text_field="text",
max_length=512
)
trainer = SFTTrainer(
model=model,
train_dataset=dataset_light,
processing_class=tokenizer,
args=training_arguments,
# Leave this out for regular SFT
#peft_config=peft_config,
)
# Train model
trainer.train()
# Save QLoRA weights
trainer.model.save_pretrained("Llama-1B-Archelec-LoRA")

Tokenizing train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


KeyboardInterrupt: 